# 03 — Tune Split 調參 → Test Split 定稿指標 + 失敗案例

**用途**:讀 M3 存在 Drive 的 keypoint cache(不需重跑推論)→ 在 **tune split**(10 falls + 13 adls)
網格搜尋規則引擎閾值 → 選出勝出設定 → 用同一組設定在 **test split**(20 falls + 27 adls,
n/s 兩模型各評一次)算出最終 precision/recall/F1 → 列出失敗案例(FP/FN)供下一步失敗分析。

**這本 notebook 全程只需 CPU**(規則引擎與模型推論解耦,只讀 cache,不碰 torch/ultralytics),
**可以選 CPU-only runtime**,不用消耗 GPU 額度。預期跑完全程約 10-15 分鐘(網格搜尋本身
約 9-10 分鐘,因為第二輪擴大了網格)。

**這是第二輪調參**(第一輪結果 4 個閾值全部卡在網格邊緣,方法論上代表範圍切太窄,
需要往更敏感的方向擴大重找;過程中也發現一個結構性 bug 已經修好,詳見下方)。

**調參設計(面試常被問的地方)**:
- **只調規則引擎階段仍可調的參數**:`kpt_conf_min`(關鍵點可見門檻)、`v_fall_enter`、
  `theta_lying_enter`(連動 `theta_upright_exit` 維持遲滯間距)、`t_confirm_fallen_s`、
  以及這輪新加入的 `window_confirm_s`(躺姿投票時窗)。`model.conf`(偵測信心門檻)
  在 M3 extract 階段就已經烘進 cache,無法事後模擬,不在這個網格裡。
- **網格範圍依第一輪結果擴大**:第一輪每個參數的勝出值都卡在候選範圍最邊緣,
  這次往「更敏感」的方向擴大(同時砍掉第一輪明顯較差的高值,把預算留給
  真正有機會的區間),但每個參數仍只留 3-4 個候選值:tune split 只有 23 支
  影片,組合太多容易在小樣本上過擬合出「剛好適合這 23 支」而非真正較好的設定。
- **新增 `window_confirm_s` 這個維度的原因**:第一輪失敗分析發現多支跌倒影片
  是「FALLING 觸發、姿態也已達標,但 track 在時窗投票還沒累積夠樣本前就消失」
  (pose 模型對躺姿的已知弱點,常見整段掉偵測)。已經在 `state_machine.py` 加了
  結構性修正(track 消失當下若最後一次觀測已符合躺姿,即使投票視窗沒跑完也收尾
  為一次事件);這裡額外把時窗本身也納入網格,看縮小視窗是否能讓更多案例在
  track 消失前就先確認完成。
- **選擇準則不變:recall 優先、precision ≥ 0.5 才列入候選**。跌倒偵測漏判的代價
  (沒人去查看、錯過黃金救援時間)高於多一次假警報,因此以 recall 為主要目標;
  但設下限避免選到「全部都報跌倒」這種退化解。
- **n 模型負責調參,n/s 都在 test split 用同一組凍結閾值評估**:`yolo26n-pose`
  是本專案預設(速度優先),閾值以它的 tune 結果為準;`yolo26s-pose` 沿用同一組
  閾值在 test split 跑一次,作為模型對照(而非重新調參)。

跑完後把最後一格印出的 JSON 摘要貼回對話。


In [ ]:
import os
if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        !git clone -q https://github.com/tun0000/fall-detection-pose.git
    %cd fall-detection-pose
!pip -q install -e "." pytest

# 同 notebook 01/02 的坑:pip install -e 之後 site.py 不會自動重新掃描 .pth,
# 這裡直接跑會 ModuleNotFoundError。reload(site) + 手動加 src/ 到 sys.path 雙重保險。
import importlib
import site
import sys

importlib.reload(site)
sys.path.insert(0, os.path.abspath("src"))

import fall_detection
print('fall_detection', fall_detection.__version__)


In [ ]:
# 規則引擎 + 調參邏輯單元測試(純 CPU,~10 秒):必須全綠
!python -m pytest -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fall-detection-pose'
DATA_DIR = f'{DRIVE_ROOT}/data/urfd'
CACHE_DIR = f'{DRIVE_ROOT}/cache'
OUT_DIR = f'{DRIVE_ROOT}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('沿用 notebook 02 的 Drive 路徑:', DRIVE_ROOT)


In [ ]:
# 讀取 tune/test 名單、GT 事件、以及 yolo26n-pose 的全部 70 支 cache
from fall_detection.eval.splits import load_splits
from fall_detection.eval.ground_truth import load_gt_events
from fall_detection.io.cache import read_cache
from fall_detection.io.urfd import all_sequences, adl_sequences

splits = load_splits('eval/splits.yaml')
gt_falls = load_gt_events(f'{DATA_DIR}/urfall-cam0-falls.csv')

sequences = all_sequences()
adl_seqs = set(adl_sequences())
tune_seqs = splits['tune']['falls'] + splits['tune']['adls']
test_seqs = splits['test']['falls'] + splits['test']['adls']
print(f'tune: {len(tune_seqs)} 支, test: {len(test_seqs)} 支')

cache_n = {seq: read_cache(f'{CACHE_DIR}/yolo26n-pose/{seq}.parquet') for seq in sequences}
print(f'已載入 yolo26n-pose cache:{len(cache_n)} 支')


In [ ]:
# tune split 網格調參(純 CPU,3x4x3x3x3=324 組 x 23 支影片,約 9-10 分鐘)
from fall_detection.config import load_config
from fall_detection.eval.report import grid_search, make_param_grid, select_best

base_cfg = load_config('config.yaml')

grid = make_param_grid(
    kpt_conf_min_values=[0.15, 0.25, 0.35],
    v_fall_enter_values=[0.6, 0.8, 1.0, 1.3],
    theta_lying_enter_values=[40, 45, 50],
    t_confirm_fallen_s_values=[0.3, 0.5, 0.7],
    window_confirm_s_values=[0.15, 0.25, 0.4],
)
print(f'網格組合數:{len(grid)}')

results = grid_search(cache_n, tune_seqs, adl_seqs, gt_falls, base_cfg, grid, tol_s=0.5)
print(f'已在 tune split({len(tune_seqs)} 支影片)評估 {len(results)} 組設定')


In [ ]:
# 挑選規則:recall 優先、precision >= 0.5(避免「全部都報」的退化解)
# 詳見 eval/report.py 的 select_best docstring
best = select_best(results, min_precision=0.5)
print('=== 勝出設定(tune split) ===')
print('params:', best['params'])
print('tune metrics:', {k: v for k, v in best['metrics'].items() if k != 'per_video'})


In [ ]:
# 用勝出設定在 test split(47 支,n/s 各評一次)——這是 README 主表數字
from fall_detection.eval.matching import evaluate_videos
from fall_detection.eval.report import _apply_params, build_video_dicts

frozen_cfg = _apply_params(base_cfg, best['params'])

test_videos_n = build_video_dicts(cache_n, test_seqs, adl_seqs, gt_falls, frozen_cfg)
metrics_test_n = evaluate_videos(test_videos_n, tol_s=0.5)

cache_s = {seq: read_cache(f'{CACHE_DIR}/yolo26s-pose/{seq}.parquet') for seq in sequences}
test_videos_s = build_video_dicts(cache_s, test_seqs, adl_seqs, gt_falls, frozen_cfg)
metrics_test_s = evaluate_videos(test_videos_s, tol_s=0.5)

print('=== test split: yolo26n-pose ===')
print({k: v for k, v in metrics_test_n.items() if k != 'per_video'})
print('=== test split: yolo26s-pose ===')
print({k: v for k, v in metrics_test_s.items() if k != 'per_video'})


In [ ]:
# 列出 test split(yolo26n-pose)的 FP/FN 案例,供下一步失敗分析挑選
from fall_detection.eval.report import list_failure_cases

fail_n = list_failure_cases(metrics_test_n)
print(f"FP 案例({len(fail_n['fp_cases'])}):")
for c in fail_n['fp_cases']:
    print(' ', c['name'], c['fp_intervals'])
print(f"FN 案例({len(fail_n['fn_cases'])}):")
for c in fail_n['fn_cases']:
    print(' ', c['name'], c['fn_intervals'])


In [ ]:
import json

def _compact(m):
    return {k: v for k, v in m.items() if k != 'per_video'}

report = {
    'winning_params': best['params'],
    'tune_metrics': _compact(best['metrics']),
    'test_metrics_yolo26n_pose': _compact(metrics_test_n),
    'test_metrics_yolo26s_pose': _compact(metrics_test_s),
    'failure_cases_yolo26n_pose': {
        'fp': [{'name': c['name'], 'intervals': c['fp_intervals']} for c in fail_n['fp_cases']],
        'fn': [{'name': c['name'], 'intervals': c['fn_intervals']} for c in fail_n['fn_cases']],
    },
}

# 完整版(含每支影片的 preds/gts 明細)存 Drive 備查;貼回對話的是精簡版
with open(f'{OUT_DIR}/m4_full.json', 'w', encoding='utf-8') as f:
    json.dump(
        {'best': best, 'metrics_test_n': metrics_test_n, 'metrics_test_s': metrics_test_s},
        f, ensure_ascii=False, indent=2,
    )

print('=== M4 摘要(請貼回對話) ===')
print(json.dumps(report, ensure_ascii=False, indent=2))


## 失敗案例分析(2 FP + 1 FN)

挑 3 個有代表性的案例畫特徵時序圖(θ/寬高比/h_hip/v_norm 疊閾值虛線,
紅色區塊 = GT 跌倒區間,藍色區塊 = 預測事件區間):

- **FN — `fall-21`**:track 在跌倒訊號真正成形前就整個消失(v_norm 一路爬升,
  最後一幀才 0.9,還沒跨過門檻資料就沒了)。
- **FP — `adl-34`**:URFD 的 ADL 集合刻意包含的「主動躺下」動作,姿態跟真的
  跌倒臥地在幾何上難以區分,評估協定又規定 ADL 影片一律零 GT。
- **FP — `fall-08`**:這支其實「是」真的跌倒影片,但預測的時間窗跟 GT 對不上
  (配對協定判定不重疊),是跟前一個 FP 完全不同的失敗型態。

跑完這幾格,直接把圖(用瀏覽器截圖或右鍵存檔)貼回對話,我會寫失敗分析文字。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from fall_detection.rules import run_engine


def plot_case(seq, gt_interval=None, pred_intervals=None, title=None):
    df, meta = cache_n[seq]
    events, debug = run_engine(df, meta.fps, frozen_cfg, collect_debug=True)
    dbg = pd.DataFrame(debug)
    if dbg.empty:
        print(f'{seq}: 無偵測資料')
        return

    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    axes[0].plot(dbg['t_s'], dbg['theta_deg'])
    axes[0].axhline(frozen_cfg.rules.theta_lying_enter, color='r', ls='--', label='theta_lying_enter')
    axes[0].axhline(frozen_cfg.rules.theta_upright_exit, color='g', ls='--', label='theta_upright_exit')
    axes[0].set_ylabel('theta_deg'); axes[0].legend(fontsize=8, loc='upper left')

    axes[1].plot(dbg['t_s'], dbg['bbox_aspect'])
    axes[1].axhline(frozen_cfg.rules.r_lying, color='r', ls='--', label='r_lying')
    axes[1].set_ylabel('bbox_aspect'); axes[1].legend(fontsize=8, loc='upper left')

    axes[2].plot(dbg['t_s'], dbg['h_hip'])
    axes[2].axhline(frozen_cfg.rules.h_hip_lying, color='r', ls='--', label='h_hip_lying')
    axes[2].set_ylabel('h_hip'); axes[2].legend(fontsize=8, loc='upper left')

    axes[3].plot(dbg['t_s'], dbg['v_norm'])
    axes[3].axhline(frozen_cfg.rules.v_fall_enter, color='r', ls='--', label='v_fall_enter')
    axes[3].set_ylabel('v_norm'); axes[3].set_xlabel('t (s)'); axes[3].legend(fontsize=8, loc='upper left')

    for ax in axes:
        if gt_interval:
            ax.axvspan(gt_interval[0], gt_interval[1], color='red', alpha=0.15)
        for p in (pred_intervals or []):
            ax.axvspan(p[0], p[1], color='blue', alpha=0.15)

    fig.suptitle(title or seq)
    plt.tight_layout()
    plt.show()
    print(f'{seq}: events={events}')


g21 = gt_falls['fall-21']
plot_case('fall-21', gt_interval=(g21[0] / 30.0, g21[1] / 30.0),
          title='FN: fall-21(紅=GT,track 在訊號成形前消失,無預測)')

plot_case('adl-34', pred_intervals=[(2.667, 6.3)],
          title='FP: adl-34(ADL 刻意躺下;藍=預測,無 GT)')

g08 = gt_falls.get('fall-08')
plot_case('fall-08', gt_interval=(g08[0] / 30.0, g08[1] / 30.0) if g08 else None,
          pred_intervals=[(2.367, 3.0)],
          title='FP: fall-08(真實跌倒影片,紅=GT、藍=預測,時間對不上)')
